# CMIP6-decadal daily `psl` full-year subset over Europe

This notebook extends the August 1962 error reproduction to request complete years of daily CMIP6-decadal sea-level pressure (`psl`) over Europe. It concatenates the same ten EC-Earth3 realizations initialized in 1961.

The default request covers the complete 1962 calendar year. A multi-year 1962–1968 variant is prepared below and can be selected with one assignment. The returned data are deliberately **not** opened with `resp.datasets()`.


## Time and area parameters

Rook's `area` parameter uses `west,south,east,north`. The bounding box below covers Europe from 10°W to 35°E and 30°N to 70°N. Rook handles the longitude conversion for datasets that use `0…360` coordinates.


In [1]:
EUROPE_BBOX = "-10,30,35,70"
SINGLE_YEAR_TIME = "1962/1962"
MULTI_YEAR_TIME = "1962/1968"


## Complete-year WPS workflow

Unlike the original August-only request, this subset has no `time_components` filter, so it retains every day in 1962.


In [2]:
request = {
    "inputs": {
        "psl": [
            "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r10i1p1f1.day.psl.gr.v20201216",
            "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r1i1p1f1.day.psl.gr.v20201215",
            "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r2i1p1f1.day.psl.gr.v20201215",
            "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r3i1p1f1.day.psl.gr.v20201215",
            "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r4i1p1f1.day.psl.gr.v20201216",
            "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r5i1p1f1.day.psl.gr.v20201216",
            "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r6i1p1f1.day.psl.gr.v20201216",
            "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r7i1p1f1.day.psl.gr.v20201216",
            "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r8i1p1f1.day.psl.gr.v20201216",
            "c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r9i1p1f1.day.psl.gr.v20201216",
        ]
    },
    "steps": {
        "concat_psl_1": {
            "run": "concat",
            "in": {"collection": "inputs/psl", "dims": "realization"},
        },
        "subset_psl_1": {
            "run": "subset",
            "in": {
                "collection": "concat_psl_1/output",
                "time": SINGLE_YEAR_TIME,
                "area": EUROPE_BBOX,
            },
        },
    },
    "outputs": {"output": "subset_psl_1/output"},
    "doc": "workflow",
}

request


{'inputs': {'psl': ['c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r10i1p1f1.day.psl.gr.v20201216',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r1i1p1f1.day.psl.gr.v20201215',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r2i1p1f1.day.psl.gr.v20201215',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r3i1p1f1.day.psl.gr.v20201215',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r4i1p1f1.day.psl.gr.v20201216',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r5i1p1f1.day.psl.gr.v20201216',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r6i1p1f1.day.psl.gr.v20201216',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r7i1p1f1.day.psl.gr.v20201216',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r8i1p1f1.day.psl.gr.v202

## Prepared multi-year variant

This copy keeps the inputs, concat operation, and European bounding box unchanged while extending the subset through 1968. Choose which request to submit by changing `selected_request`.


In [3]:
from copy import deepcopy

multi_year_request = deepcopy(request)
multi_year_request["steps"]["subset_psl_1"]["in"]["time"] = MULTI_YEAR_TIME

# selected_request = request
selected_request = multi_year_request

selected_request["steps"]["subset_psl_1"]["in"]


{'collection': 'concat_psl_1/output',
 'time': '1962/1968',
 'area': '-10,30,35,70'}

## Build the equivalent Rooki workflow

Importing Rooki contacts the configured WPS service. Change `ROOK_URL` if the reproduction should run against another deployment.


In [4]:
import json
import os

os.environ["ROOK_URL"] = "http://rook.dkrz.de/wps"

from rooki import operators as ops


In [5]:
psl = ops.Input("psl", selected_request["inputs"]["psl"])
concat = ops.Concat(psl, dims="realization")
subset_parameters = selected_request["steps"]["subset_psl_1"]["in"]
subset = ops.Subset(
    concat,
    time=subset_parameters["time"],
    area=subset_parameters["area"],
)

serialized_request = json.loads(subset._serialise())
serialized_request


{'inputs': {'psl': ['c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r10i1p1f1.day.psl.gr.v20201216',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r1i1p1f1.day.psl.gr.v20201215',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r2i1p1f1.day.psl.gr.v20201215',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r3i1p1f1.day.psl.gr.v20201215',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r4i1p1f1.day.psl.gr.v20201216',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r5i1p1f1.day.psl.gr.v20201216',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r6i1p1f1.day.psl.gr.v20201216',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r7i1p1f1.day.psl.gr.v20201216',
   'c3s-cmip6-decadal.DCPP.EC-Earth-Consortium.EC-Earth3.dcppA-hindcast.s1961-r8i1p1f1.day.psl.gr.v202

## Reproduce the failure

The next cell submits the selected complete-year workflow. Run it only against the deployment being investigated.


In [6]:
from time import perf_counter

started_at = perf_counter()
resp = subset.orchestrate()
elapsed_seconds = perf_counter() - started_at

print(f"Orchestration time: {elapsed_seconds:.1f} seconds")
resp.ok, resp.status


Orchestration time: 528.6 seconds


(True, 'ProcessSucceeded')

## Inspect the response without loading data

Displaying the response preserves failure details for diagnosis. On success, report the output size and URLs without downloading or opening the NetCDF result.


In [7]:
resp


Metalink URL: http://rook7.cloud.dkrz.de:80/outputs/rook/9846e926-9c1d-11f1-b2d2-fa163eb671ca/input.meta4, num files: 7

In [8]:
if resp.ok:
    print(f"Output size: {resp.size_in_mb:.2f} MiB")
    print("Output URLs (not downloaded):")
    for url in resp.download_urls():
        print(url)


Output size: 264.40 MiB
Output URLs (not downloaded):
http://rook7.cloud.dkrz.de:80/outputs/rook/d2c3db6c-9c1e-11f1-8aa1-fa163eb671ca/psl_day_EC-Earth3_dcppA-hindcast_r10i1p1f1_gr_19620101-19621231.nc
http://rook7.cloud.dkrz.de:80/outputs/rook/d2c3ef76-9c1e-11f1-8aa1-fa163eb671ca/psl_day_EC-Earth3_dcppA-hindcast_r10i1p1f1_gr_19630101-19631231.nc
http://rook7.cloud.dkrz.de:80/outputs/rook/d2c3fc8c-9c1e-11f1-8aa1-fa163eb671ca/psl_day_EC-Earth3_dcppA-hindcast_r10i1p1f1_gr_19640101-19641231.nc
http://rook7.cloud.dkrz.de:80/outputs/rook/d2c40768-9c1e-11f1-8aa1-fa163eb671ca/psl_day_EC-Earth3_dcppA-hindcast_r10i1p1f1_gr_19650101-19651231.nc
http://rook7.cloud.dkrz.de:80/outputs/rook/d2c411a4-9c1e-11f1-8aa1-fa163eb671ca/psl_day_EC-Earth3_dcppA-hindcast_r10i1p1f1_gr_19660101-19661231.nc
http://rook7.cloud.dkrz.de:80/outputs/rook/d2c41cb2-9c1e-11f1-8aa1-fa163eb671ca/psl_day_EC-Earth3_dcppA-hindcast_r10i1p1f1_gr_19670101-19671231.nc
http://rook7.cloud.dkrz.de:80/outputs/rook/d2c427fc-9c1e-11f1-8a